# Real-time Data Processing with Azure Databricks (and Event Hubs)

This notebook demonstrates the below architecture to build real-time data pipelines.
![Solution Architecture](https://raw.githubusercontent.com/malvik01/Real-Time-Streaming-with-Azure-Databricks/main/Azure%20Solution%20Architecture.png)


- Data Sources: Streaming data from IoT devices or social media feeds. (Simulated in Event Hubs)
- Ingestion: Azure Event Hubs for capturing real-time data.
- Processing: Azure Databricks for stream processing using Structured Streaming.
- Storage: Processed data stored Azure Data Lake (Delta Format).
- Visualisation: Data visualized using Power BI.


### Azure Services Required
- Databricks Workspace (Unity Catalog enabled)
- Azure Data Lake Storage (Premium)
- Azure Event Hub (Basic Tier)

### Azure Databricks Configuration Required
- Single Node Compute Cluster: `13.3 LTS (includes Apache Spark 3.3.2, Scala 2.12)`
- Maven Library installed on Compute Cluster: `com.microsoft.azure:azure-eventhubs-spark_2.12:2.3.22`

Importing the libraries.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

The code block below creates the catalog and schemas for our solution. 

The approach utilises a multi-hop data storage architecture (medallion), consisting of bronze, silver, and gold schemas within a 'streaming' catalog. 

In [0]:
try:
    spark.sql("use catalog demo;")
except:
    print('check if catalog already exists')

try:
    spark.sql("create schema demo.bronze;")
except:
    print('check if bronze schema already exists')

try:
    spark.sql("create schema demo.silver")
except:
    print('check if silver schema already exists')

try:
    spark.sql("create schema demo.gold;")
except:
    print('check if gold schema already exists')

#### Bronze Layer

Set up Azure Event hubs connection string.

In [0]:
# Config
# Replace with your Event Hub namespace, name, and key
connectionString = "Endpoint=sb://eh-namesp.servicebus.windows.net/;SharedAccessKeyName=databricks;SharedAccessKey=hoPwvVl5ggzUvMQdPhLjzO9GgsE3/6DwJ+AEhBFfJTk=;EntityPath=eh-tqt"
eventHubName = "eh-tqt"

ehConf = {
  'eventhubs.connectionString' : sc._jvm.org.apache.spark.eventhubs.EventHubsUtils.encrypt(connectionString),
  'eventhubs.eventHubName': eventHubName
}

Reading and writing the stream to the bronze layer.

In [0]:
# Reading stream: Load data from Azure Event Hub into DataFrame 'df' using the previously configured settings
df = spark.readStream \
    .format("eventhubs") \
    .options(**ehConf) \
    .load() \

# Displaying stream: Show the incoming streaming data for visualization and debugging purposes
df.display()
# Writing stream: Persist the streaming data to a Delta table 'streaming.bronze.weather' in 'append' mode with checkpointing
df.writeStream \
    .option("checkpointLocation", "/mnt/streaming/bronze/weather") \
    .outputMode("append") \
    .format("delta") \
    .toTable("demo.bronze.weather")

body,partition,offset,sequenceNumber,enqueuedTime,publisher,partitionKey,properties,systemProperties
ew0KICAgICJ0ZW1wZXJhdHVyZSI6IDE4LA0KICAgICJodW1pZGl0eSI6IDYwLA0KICAgICJ3aW5kU3BlZWQiOiAxMCwNCiAgICAid2luZERpcmVjdGlvbiI6ICJTVyIsDQogICAgInByZWNpcGl0YXRpb24iOiAwLA0KICAgICI= (truncated),1,4294968064,3,2026-04-18T17:50:21.341Z,null,null,Map(),"Map(x-opt-sequence-number-epoch -> -1, correlation-id -> 2cdc9b37-2abd-41a5-aeec-48b54b662bc6, message-id -> EHExplorer-64d739fd-48da-40b1-8b35-186fc45f44aa, content-type -> application/json)"
ew0KICAgICJ0ZW1wZXJhdHVyZSI6IDI4LA0KICAgICJodW1pZGl0eSI6IDYwLA0KICAgICJ3aW5kU3BlZWQiOiAxMCwNCiAgICAid2luZERpcmVjdGlvbiI6ICJOVyIsDQogICAgInByZWNpcGl0YXRpb24iOiAwLA0KICAgICI= (truncated),1,4294968448,4,2026-04-18T17:58:18.129Z,null,null,Map(),"Map(x-opt-sequence-number-epoch -> -1, correlation-id -> 7915869c-eaba-4bad-a02e-3766daa3912d, message-id -> EHExplorer-e574fbed-ca56-4082-b96b-3a82299ecfff, content-type -> application/json)"
ew0KICAgICJ0ZW1wZXJhdHVyZSI6IDQwLA0KICAgICJodW1pZGl0eSI6IDgwLA0KICAgICJ3aW5kU3BlZWQiOiAxMCwNCiAgICAid2luZERpcmVjdGlvbiI6ICJzVyIsDQogICAgInByZWNpcGl0YXRpb24iOiAwLA0KICAgICI= (truncated),1,4294968824,5,2026-04-18T18:00:01.353Z,null,null,Map(),"Map(x-opt-sequence-number-epoch -> -1, correlation-id -> 28b3f1fd-653c-4884-bbca-df64fbd818f9, message-id -> EHExplorer-aa9042f3-d441-4da3-90d5-40c73551def6, content-type -> application/json)"
ew0KICAgICJ0ZW1wZXJhdHVyZSI6IDEwLA0KICAgICJodW1pZGl0eSI6IDYwLA0KICAgICJ3aW5kU3BlZWQiOiAxMCwNCiAgICAid2luZERpcmVjdGlvbiI6ICJTVyIsDQogICAgInByZWNpcGl0YXRpb24iOiAwLA0KICAgICI= (truncated),0,4294968832,5,2026-04-18T18:14:24.186Z,null,null,Map(),"Map(x-opt-sequence-number-epoch -> -1, correlation-id -> 57cb09eb-4602-487f-b517-54126d8e2953, message-id -> EHExplorer-36819404-63b2-4a7f-8641-8f104992688c, content-type -> application/json)"


#### Silver Layer

Defining the schema for the JSON object.

In [0]:
# Defining the schema for the JSON object

json_schema = StructType([
    StructField("temperature", IntegerType()),
    StructField("humidity", IntegerType()),
    StructField("windSpeed", IntegerType()),
    StructField("windDirection", StringType()),
    StructField("precipitation", IntegerType()),
    StructField("conditions", StringType())
])

Reading, transforming and writing the stream from the bronze to the silver layer.

In [0]:
# Reading and Transforming: Load streaming data from the 'streaming.bronze.weather' Delta table, cast 'body' to string, parse JSON, and select specific fields
df = spark.readStream\
    .format("delta")\
    .table("demo.bronze.weather")\
    .withColumn("body", col("body").cast("string"))\
    .withColumn("body",from_json(col("body"), json_schema))\
    .select("body.temperature", "body.humidity", "body.windSpeed", "body.windDirection", "body.precipitation", "body.conditions", col("enqueuedTime").alias('timestamp'))

# Displaying stream: Visualize the transformed data in the DataFrame for verification and analysis
df.display()

# Writing stream: Save the transformed data to the 'streaming.silver.weather' Delta table in 'append' mode with checkpointing for data reliability
df.writeStream\
    .option("checkpointLocation", "/mnt/streaming/silver/weather")\
    .outputMode("append")\
    .format("delta")\
    .toTable("demo.silver.weather")

temperature,humidity,windSpeed,windDirection,precipitation,conditions,timestamp
20,60,10,NW,0,Partly Cloudy,2026-04-18T17:24:18.567Z
20,60,10,NW,0,Partly Cloudy,2026-04-18T17:21:59.214Z
20,60,10,NW,0,Partly Cloudy,2026-04-18T17:27:45.553Z
30,60,10,NW,0,Partly Cloudy,2026-04-18T17:27:52.808Z
30,60,10,NW,0,Partly Cloudy,2026-04-18T17:25:19.282Z
18,60,10,SW,0,Partly Cloudy,2026-04-18T17:50:21.341Z
28,60,10,NW,0,Sunny,2026-04-18T17:58:18.129Z
40,80,10,sW,0,Sunny,2026-04-18T18:00:01.353Z
10,60,10,SW,0,Cloudy,2026-04-18T18:14:24.186Z


#### Gold Layer

Reading, aggregating and writing the stream from the silver to the gold layer.

In [0]:
# Aggregating Stream: Read from 'streaming.silver.weather', apply watermarking and windowing, and calculate average weather metrics
df = spark.readStream\
    .format("delta")\
    .table("demo.silver.weather")\
    .withWatermark("timestamp", "5 minutes") \
    .groupBy(window("timestamp", "5 minutes")) \
    .agg(avg("temperature").alias('temperature'), avg("humidity").alias('humidity'), avg("windSpeed").alias('windSpeed'), avg("precipitation").alias('precipitation'))\
	.select('window.start', 'window.end', 'temperature', 'humidity', 'windSpeed', 'precipitation')

# Displaying Aggregated Stream: Visualize aggregated data for insights into weather trends
df.display()

# Writing Aggregated Stream: Store the aggregated data in 'streaming.gold.weather_aggregated' with checkpointing for data integrity
df.writeStream\
    .option("checkpointLocation", "/mnt/streaming/weather_summary")\
    .outputMode("append")\
    .format("delta")\
    .toTable("demo.gold.weather_summary")

start,end,temperature,humidity,windSpeed,precipitation
2026-04-18T17:25:00Z,2026-04-18T17:30:00Z,26.666666666666668,60.0,10.0,0.0
2026-04-18T17:55:00Z,2026-04-18T18:00:00Z,28.0,60.0,10.0,0.0
2026-04-18T18:00:00Z,2026-04-18T18:05:00Z,40.0,80.0,10.0,0.0
2026-04-18T17:50:00Z,2026-04-18T17:55:00Z,18.0,60.0,10.0,0.0
2026-04-18T18:10:00Z,2026-04-18T18:15:00Z,10.0,60.0,10.0,0.0
2026-04-18T17:20:00Z,2026-04-18T17:25:00Z,20.0,60.0,10.0,0.0
